In [1]:
!pip install -qU langchain langchain-chroma langchain-community langchain-huggingface chromadb pypdf sentence-transformers

In [2]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain-chroma import Chroma

pdf_folder_path = "/kaggle/input/datasets/jiyajain23/aim-docs" 
chroma_save_path = "/kaggle/working/chroma_db/"

print("Step 1: Loading Documents...")

loader = PyPDFDirectoryLoader(pdf_folder_path)
raw_documents = loader.load()
print(f"-> Loaded {len(raw_documents)} pages from the PDFs.")

print("\nStep 2: Chunking Text...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200, # 200 char overlap ensures context isn't lost between chunks
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(raw_documents)
print(f"-> Split documents into {len(chunks)} searchable chunks.")

print("\nStep 3: Initializing Embedding Model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("\nStep 4: Creating Vector Database (This may take a few minutes)...")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=chroma_save_path
)

print(f"\n Phase 1 Complete! Vector database saved to {chroma_save_path}")
print("You can now download this folder from the Kaggle Output section later.")

/tmp/ipykernel_261/220948218.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Step 1: Loading Documents...
-> Loaded 587 pages from the PDFs.

Step 2: Chunking Text...
-> Split documents into 1491 searchable chunks.

Step 3: Initializing Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Step 4: Creating Vector Database (This may take a few minutes)...

 Phase 1 Complete! Vector database saved to /kaggle/working/chroma_db/
You can now download this folder from the Kaggle Output section later.


In [6]:
!pip install -qU langchain-groq langchain-classic rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.4 MB/s eta 0:00:00 0:00:01


In [26]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# BM25 needs the raw chunk list, not the vector store
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

vector_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# Weighted combination: tune weights based on how much you trust keyword vs semantic match
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma(persist_directory="/kaggle/working/chroma_db/", embedding_function=embeddings)
# 2. Setup Groq LLM
groq_api_key = UserSecretsClient().get_secret("groq-api-key")
llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0.2)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.messages import HumanMessage, AIMessage

# 1. Prompt to reformulate follow-up questions using chat history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and the latest user question which might reference "
               "context in the chat history, reformulate it into a standalone question. "
               "Do NOT answer the question, just reformulate it if needed, otherwise return it as is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

history_aware_retriever = create_history_aware_retriever(llm, hybrid_retriever, contextualize_prompt)

# 2. QA prompt that incorporates context and history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based only on the following context. "
               "If the answer isn't in the context, say you don't know.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

conv_combine_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, conv_combine_chain)

# 3. Memory tracking helper
chat_history = []

def ask(question):
    result = conversational_rag_chain.invoke({"input": question, "chat_history": chat_history})
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result["answer"]))
    return result["answer"]

# 4. Test follow-up queries!
print(ask("What is an Abnormal Runway Contact (ARC)?"))
print("-----next query-----")
print(ask("Does that include helicopters?"))

An Abnormal Runway Contact (ARC) is any landing or takeoff involving abnormal runway or landing surface contact. This includes events such as:

- Hard/heavy landings
- Long/fast landings
- Off-center landings
- Crabbed landings
- Nose wheel first touchdown
- Tail strikes
- Wingtip/nacelle strikes.
-----next query-----
I don't know. The context provided does not mention helicopters.
